# NB51 — 63k Genis Veri: Sizinti Denetimi & Saglamlik Kontrolu

Plan: `docs/PLAN_63K_ENTEGRASYON.md` ADIM 2.

> **Bu adim atlanamaz.** 777 sutunlu bir veride etiketi `clinvar__sig`'den turetmek
> sizintiyi neredeyse davet eder. Eski projenin en pahali hatasi (`v1_leaked` klasoru)
> buydu.

NB50'nin urettigi `missense_63k.parquet` (60.970 satir, 538 sutun, sizinti-temiz
oldugu iddia edilen) burada 5 kontrolden gecirilir:
1. Hizli sinyal testi (LGBM, CV F1>0.97 kirmizi bayrak)
2. Tek-sutun AUC taramasi (>0.95 elle incele -- NB50'de 30 sutun bulundu)
3. Meta-predictor ablasyonu (dahil/haric CV F1 farki)
4. Gen-ezberi testi (GroupKFold vs StratifiedKFold)
5. Duplicate satir kontrolu

In [1]:
# Cell 1: Imports & Config
import os, sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import SEED
from src import columns_63k as C63

np.random.seed(SEED)

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef
import lightgbm as lgb

PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', '63k_genis')
RESULTS_PREP_DIR = os.path.join(PROJECT_ROOT, 'results', 'v31_63k_prep')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v32_63k_audit')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MISSENSE_PARQUET = os.path.join(PARQUET_DIR, 'missense_63k.parquet')
df = pd.read_parquet(MISSENSE_PARQUET)
print('missense_63k.parquet yuklendi:', df.shape)

with open(os.path.join(RESULTS_PREP_DIR, 'nb50_column_lists.json')) as f:
    col_lists = json.load(f)

meta_score_cols = [c for c in col_lists['meta_score_cols'] if c in df.columns]
meta_rankscore_cols = [c for c in col_lists.get('meta_rankscore_cols', []) if c in df.columns]
meta_pred_cols = [c for c in col_lists['meta_pred_cols'] if c in df.columns]
numeric_cols = [c for c in col_lists['numeric_cols'] if c in df.columns]
categorical_cols = [c for c in col_lists['categorical_cols'] if c in df.columns]
binary_cols = [c for c in col_lists['binary_cols'] if c in df.columns]

feature_cols = numeric_cols + categorical_cols + binary_cols
print(f'Feature adaylari: numeric={len(numeric_cols)}, categorical={len(categorical_cols)}, binary={len(binary_cols)}, toplam={len(feature_cols)}')
print(f'Meta-predictor (ayri tutulan): score={len(meta_score_cols)}, rankscore={len(meta_rankscore_cols)}, pred={len(meta_pred_cols)}')

y = df['Label'].astype(int)
prevalence = y.mean()
floor_f1 = C63.floor_f1(prevalence)
print(f'Prevalans={prevalence:.4f}, floor-F1={floor_f1:.4f}')

missense_63k.parquet yuklendi: (60970, 533)
Feature adaylari: numeric=358, categorical=164, binary=0, toplam=522
Meta-predictor (ayri tutulan): score=33, rankscore=24, pred=6
Prevalans=0.3727, floor-F1=0.5430


## Adim 2.1 — Hizli Sinyal Testi

Temizlenmis feature seti ile default hiperparametreli LGBM, 5-fold stratified CV.
**Kirmizi bayrak: CV F1 > 0.97 veya AUC > 0.995.**

In [2]:
# Cell 2: Hizli sinyal testi -- LGBM default, 5-fold CV, TUM feature'lar (meta dahil, ilk tarama)
def prepare_lgbm_frame(frame, cols, cat_cols):
    X = frame[cols].copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].astype('category')
    return X

cat_cols_present = [c for c in categorical_cols if c in feature_cols]

def run_cv_lgbm(X, y, groups=None, n_splits=5, cat_features='auto'):
    if groups is None:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        split_iter = splitter.split(X, y)
    else:
        splitter = GroupKFold(n_splits=n_splits)
        split_iter = splitter.split(X, y, groups=groups)

    f1s, aucs, mccs = [], [], []
    for train_idx, val_idx in split_iter:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, verbosity=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=cat_features)
        proba = model.predict_proba(X_val)[:, 1]
        pred = (proba >= 0.5).astype(int)

        f1s.append(f1_score(y_val, pred))
        aucs.append(roc_auc_score(y_val, proba))
        mccs.append(matthews_corrcoef(y_val, pred))
    return np.array(f1s), np.array(aucs), np.array(mccs)

X_all = prepare_lgbm_frame(df, feature_cols, cat_cols_present)
f1s, aucs, mccs = run_cv_lgbm(X_all, y, cat_features=cat_cols_present)

print(f'CV F1  = {f1s.mean():.4f} +/- {f1s.std():.4f}')
print(f'CV AUC = {aucs.mean():.4f} +/- {aucs.std():.4f}')
print(f'CV MCC = {mccs.mean():.4f} +/- {mccs.std():.4f}')
print(f'Floor-F1 referansi: {floor_f1:.4f}')

RED_FLAG = f1s.mean() > 0.97 or aucs.mean() > 0.995
if RED_FLAG:
    print()
    print('!!! KIRMIZI BAYRAK: CV F1>0.97 veya AUC>0.995 -- sizinti supheli, feature importance incelenecek.')
else:
    print()
    print('Kirmizi bayrak tetiklenmedi (F1<=0.97 ve AUC<=0.995).')

CV F1  = 0.9899 +/- 0.0009
CV AUC = 0.9996 +/- 0.0001
CV MCC = 0.9840 +/- 0.0014
Floor-F1 referansi: 0.5430

!!! KIRMIZI BAYRAK: CV F1>0.97 veya AUC>0.995 -- sizinti supheli, feature importance incelenecek.


In [3]:
# Cell 3: Feature importance top-30 (kirmizi bayrak durumunda sizinti kaynagini bulmak icin;
# tetiklenmese de meta-predictor'larin agirligini gormek adina her zaman calistiriliyor)
model_full = lgb.LGBMClassifier(n_estimators=200, random_state=SEED, verbosity=-1)
model_full.fit(X_all, y, categorical_feature=cat_cols_present)

importance = pd.Series(model_full.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('Feature importance top-30:')
print(importance.head(30))

_all_meta_preview = set(meta_score_cols) | set(meta_rankscore_cols) | set(meta_pred_cols)
top30_is_meta = [c for c in importance.head(30).index if c in _all_meta_preview]
print()
print(f'Top-30 icinde meta-predictor sayisi: {len(top30_is_meta)} -> {top30_is_meta}')

importance.to_csv(os.path.join(RESULTS_DIR, 'nb51_feature_importance.csv'))

Feature importance top-30:
ditto__score                          340
mutpred1__external_protein_id         179
gnomad__af                            171
primateai__primateai_score            134
metarnn__score                        131
swissprot_domains__domain             123
alphamissense__am_pathogenicity       117
vest__score                           106
metarnn__rank_score                   103
allofus250k__gvs_max_af               101
mutpred1__mutpred_general_score        99
gmvp__score                            98
revel__score                           92
regeneron__ALL_AN                      87
mistic__score                          76
regeneron__ALL_AF                      72
ncer__score                            66
vest__pval                             65
sift__seqs                             65
bayesdel__bayesdel_addAF_score         64
cscape__score                          60
primateai__primateai_rankscore         60
mutation_assessor__score               57
dann__s

## Adim 2.2 — Tek-Sutun AUC Taramasi (NB50'nin 30 kirmizi-bayrakli sutununu siniflandir)

Her sutun icin **elle** karar: meta-predictor (zaten ayri liste, meşru ama dongusel),
meşru güçlü predictor (populasyon frekansi, konservasyon), yoksa gerçek sizinti (ID
sizintisi, transkript kalintisi vb. NB50'de gozden kacan).

In [4]:
# Cell 4: NB50'nin auc>0.95 listesini yukle, feature_cols ile kesistir, elle siniflandir
auc_series = pd.read_csv(os.path.join(RESULTS_PREP_DIR, 'nb50_univariate_auc.csv'), index_col=0).iloc[:, 0]
high_auc = auc_series[auc_series > 0.95].sort_values(ascending=False)
print(f'NB50 auc>0.95 sutun sayisi: {len(high_auc)}')
print(high_auc)

_all_meta_classify = set(meta_score_cols) | set(meta_rankscore_cols) | set(meta_pred_cols)

# Elle siniflandirma kurali:
#   - meta_score/rankscore/pred_cols icindeyse -> 'meta_predictor' (dongusel ama bilinen, A6'da ayri ele alinacak)
#   - 'gnomad' / 'thousandgenomes' / 'alfa' / 'allofus' / 'hgdp' iceriyorsa -> 'population_freq' (mesru bioloji: nadir=patojenik egilimi)
#   - 'conservation'/'phylop'/'phastcons'/'gerp' iceriyorsa -> 'conservation' (mesru)
#   - fonksiyonel deney / hastalik-spesifik literatur anotasyonu (brca1_func_assay, arrvars) -> 'functional_assay' (mesru ama az satirda dolu, dikkatli kullan)
#   - digerleri -> 'UNKNOWN_INVESTIGATE' (elle bakilmasi gereken supheli kalan)
def classify_high_auc_col(col):
    if col in _all_meta_classify:
        return 'meta_predictor'
    lc = col.lower()
    if any(k in lc for k in ['gnomad', 'thousandgenomes', 'alfa', 'allofus', 'hgdp', '_af', '_ac', '_an']):
        return 'population_freq'
    if any(k in lc for k in ['phylop', 'phastcons', 'gerp', 'conserv']):
        return 'conservation'
    if any(k in lc for k in ['func_assay', 'arrvars']):
        return 'functional_assay'
    return 'UNKNOWN_INVESTIGATE'

classification = pd.Series({c: classify_high_auc_col(c) for c in high_auc.index})
print()
print('Siniflandirma dagilimi:')
print(classification.value_counts())

unknown = classification[classification == 'UNKNOWN_INVESTIGATE']
print()
if len(unknown) > 0:
    print(f'!!! ELLE INCELE: {len(unknown)} sutun hicbir bilinen mesru kategoriye girmiyor:')
    print(high_auc.loc[unknown.index])
else:
    print('Tum yuksek-AUC sutunlar bilinen mesru kategorilere (meta-predictor / population_freq / conservation / functional_assay) dustu.')

high_auc_report = pd.DataFrame({'auc': high_auc, 'category': classification})
high_auc_report.to_csv(os.path.join(RESULTS_DIR, 'nb51_high_auc_classification.csv'))

NB50 auc>0.95 sutun sayisi: 30
ditto__score                          0.996841
metarnn__score                        0.996018
metarnn__rank_score                   0.996018
cardioboost__arrhythmias              0.995668
clinpred__rankscore                   0.994386
clinpred__score                       0.994386
bayesdel__bayesdel_addAF_rankscore    0.992926
bayesdel__bayesdel_addAF_score        0.992926
brca1_func_assay__score               0.980827
revel__score                          0.976320
revel__rankscore                      0.976320
bayesdel__bayesdel_noAF_score         0.974760
bayesdel__bayesdel_noAF_rankscore     0.974760
gmvp__score                           0.969198
gmvp__rank_score                      0.969198
vest__score                           0.968506
vest__pval                            0.968500
mistic__score                         0.966045
varity_r__varity_r                    0.964492
varity_r__varity_r_loo                0.960471
arrvars__lqt_penetrance      

## Adim 2.3 — Meta-Predictor Ablasyonu (A6'nin on hazirligi)

`LEAKY_META_PREDICTOR_SCORES/PREDS` ile ve olmadan CV F1 farkini olc. **Fark >0.05 ise
bu sutunlar "modelin isini yapiyor" demektir** -- dahil edip etmeme acikca belgelenmis
bir karar olmali.

In [5]:
# Cell 5: Meta-predictor dahil vs haric CV F1 farki
_all_meta = set(meta_score_cols) | set(meta_rankscore_cols) | set(meta_pred_cols)
non_meta_cols = [c for c in feature_cols if c not in _all_meta]
non_meta_cat_cols = [c for c in cat_cols_present if c in non_meta_cols]

X_nometa = prepare_lgbm_frame(df, non_meta_cols, non_meta_cat_cols)
f1s_nometa, aucs_nometa, mccs_nometa = run_cv_lgbm(X_nometa, y, cat_features=non_meta_cat_cols)

print('=== Meta-predictor DAHIL (Cell 2 sonucu) ===')
print(f'CV F1  = {f1s.mean():.4f} +/- {f1s.std():.4f}')
print(f'CV AUC = {aucs.mean():.4f} +/- {aucs.std():.4f}')
print()
print('=== Meta-predictor HARIC (score+rankscore+pred hepsi cikarildi) ===')
print(f'CV F1  = {f1s_nometa.mean():.4f} +/- {f1s_nometa.std():.4f}')
print(f'CV AUC = {aucs_nometa.mean():.4f} +/- {aucs_nometa.std():.4f}')

f1_delta = f1s.mean() - f1s_nometa.mean()
print()
print(f'F1 farki (dahil - haric) = {f1_delta:.4f}')
if f1_delta > 0.05:
    print('SONUC: Meta-predictor sutunlari "modelin isini yapiyor" (fark>0.05).')
    print('  -> A6 ablasyonunda dahil/haric ayrimi ACIKCA belgelenmeli, sessiz varsayilan olarak dahil edilmemeli.')
else:
    print('SONUC: Meta-predictor katkisi sinirli (fark<=0.05) -- dahil etmek daha az riskli bir varsayilan olabilir, yine de A6da dogrulanacak.')

meta_ablation_result = {
    'f1_with_meta': float(f1s.mean()), 'f1_without_meta': float(f1s_nometa.mean()),
    'f1_delta': float(f1_delta),
    'auc_with_meta': float(aucs.mean()), 'auc_without_meta': float(aucs_nometa.mean()),
}
with open(os.path.join(RESULTS_DIR, 'nb51_meta_ablation.json'), 'w') as f:
    json.dump(meta_ablation_result, f, indent=2)

=== Meta-predictor DAHIL (Cell 2 sonucu) ===
CV F1  = 0.9899 +/- 0.0009
CV AUC = 0.9996 +/- 0.0001

=== Meta-predictor HARIC (score+rankscore+pred hepsi cikarildi) ===
CV F1  = 0.9878 +/- 0.0004
CV AUC = 0.9994 +/- 0.0001

F1 farki (dahil - haric) = 0.0021
SONUC: Meta-predictor katkisi sinirli (fark<=0.05) -- dahil etmek daha az riskli bir varsayilan olabilir, yine de A6da dogrulanacak.


## Adim 2.4 — Gen-Ezberi Testi (GroupKFold vs StratifiedKFold)

`GroupKFold(groups=base__hugo)` ile ikinci bir degerlendirme. **Fark >0.10 ise model
ciddi bicimde gen ezberliyor** demektir.

In [6]:
# Cell 6: Gen-holdout (GroupKFold) vs rastgele (StratifiedKFold) F1 farki
# base__hugo feature olarak KULLANILMIYOR (zaten feature_cols'ta yok) -- sadece grup anahtari
groups = df[C63.GENE_GROUP_COL]
print('Benzersiz gen sayisi:', groups.nunique())

f1s_group, aucs_group, mccs_group = run_cv_lgbm(X_nometa, y, groups=groups, cat_features=non_meta_cat_cols)

print('=== StratifiedKFold (rastgele-split, meta-predictor haric) ===')
print(f'CV F1  = {f1s_nometa.mean():.4f} +/- {f1s_nometa.std():.4f}')
print()
print('=== GroupKFold (gen-holdout, meta-predictor haric) ===')
print(f'CV F1  = {f1s_group.mean():.4f} +/- {f1s_group.std():.4f}')

gene_gap = f1s_nometa.mean() - f1s_group.mean()
print()
print(f'Gen-ezberi farki (rastgele - gen-holdout) = {gene_gap:.4f}')
if gene_gap > 0.10:
    print('!!! SONUC: Model ciddi bicimde gen ezberliyor (fark>0.10).')
    print('  -> ADIM 3de gen-turevli her sey drop edilmis olmali (zaten base__hugo feature degil),')
    print('     regularizasyon artirilmali, gen-holdout BIRINCIL degerlendirme metrigi yapilmali.')
else:
    print('SONUC: Gen-ezberi belirgin degil (fark<=0.10) -- ama gen-holdout yine de her tabloda raporlanmali.')

gene_memorization_result = {
    'f1_random_split': float(f1s_nometa.mean()), 'f1_group_split': float(f1s_group.mean()),
    'gene_gap': float(gene_gap), 'n_unique_genes': int(groups.nunique()),
}
with open(os.path.join(RESULTS_DIR, 'nb51_gene_memorization.json'), 'w') as f:
    json.dump(gene_memorization_result, f, indent=2)

Benzersiz gen sayisi: 7207
=== StratifiedKFold (rastgele-split, meta-predictor haric) ===
CV F1  = 0.9878 +/- 0.0004

=== GroupKFold (gen-holdout, meta-predictor haric) ===
CV F1  = 0.9822 +/- 0.0009

Gen-ezberi farki (rastgele - gen-holdout) = 0.0056
SONUC: Gen-ezberi belirgin degil (fark<=0.10) -- ama gen-holdout yine de her tabloda raporlanmali.


## Adim 2.5 — Duplicate Satir Kontrolu

`base__chrom + base__pos + base__ref_base + base__alt_base` uzerinden tam kopya
kontrolu.

In [7]:
# Cell 7: Duplicate satir kontrolu
dup_key_cols = [C63.CHROM_COL, C63.POS_COL, C63.REF_COL, C63.ALT_COL]
dup_mask = df.duplicated(subset=dup_key_cols, keep=False)
n_dup_rows = dup_mask.sum()
print(f'Tam kopya varyant sayisi ({"+".join(dup_key_cols)} bazinda): {n_dup_rows}')

if n_dup_rows > 0:
    dup_groups = df[dup_mask].groupby(dup_key_cols, observed=True)['Label'].agg(['count', 'nunique'])
    conflicting_dups = dup_groups[dup_groups['nunique'] > 1]
    print(f'Bunlardan celiskili etiketli (ayni varyant, farkli Label) grup sayisi: {len(conflicting_dups)}')
    if len(conflicting_dups) > 0:
        print(conflicting_dups.head(10))
else:
    print('Duplicate satir bulunamadi.')

duplicate_row_result = {
    'n_duplicate_rows': int(n_dup_rows),
    'n_conflicting_label_groups': int(len(conflicting_dups)) if n_dup_rows > 0 else 0,
}
with open(os.path.join(RESULTS_DIR, 'nb51_duplicate_rows.json'), 'w') as f:
    json.dump(duplicate_row_result, f, indent=2)

Tam kopya varyant sayisi (base__chrom+base__pos+base__ref_base+base__alt_base bazinda): 18
Bunlardan celiskili etiketli (ayni varyant, farkli Label) grup sayisi: 0


## Sonuc: "Bu Veri Seti Temizdir Cunku ..." Raporu

In [8]:
# Cell 8: Ozet + otomatik PDF rapor
summary = {
    'cv_f1_all_features_incl_meta': float(f1s.mean()),
    'cv_auc_all_features_incl_meta': float(aucs.mean()),
    'red_flag_quick_signal': bool(RED_FLAG),
    'n_high_auc_gt_095': int(len(high_auc)),
    'n_high_auc_unknown_investigate': int(len(unknown)),
    'meta_ablation_f1_delta': float(f1_delta),
    'gene_memorization_gap': float(gene_gap),
    'n_duplicate_rows': int(n_dup_rows),
    'n_conflicting_label_groups': int(len(conflicting_dups)) if n_dup_rows > 0 else 0,
    'floor_f1': float(floor_f1),
}
with open(os.path.join(RESULTS_DIR, 'nb51_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('=== NB51 OZET ===')
for k, v in summary.items():
    print(f'{k}: {v}')

# "Temiz" verdict'i artik sadece bilinen-kategori/kirmizi-bayrak kontrolune degil,
# CV F1'in meta-predictor'lar TAMAMEN cikarilsa bile hala anormal yuksek olup
# olmadigina da bakiyor -- bu, Cell 5'te ortaya cikan asil bulgu (bkz. asagi not).
clean_verdict = (
    (not RED_FLAG)
    and (len(unknown) == 0)
    and (len(conflicting_dups) == 0 if n_dup_rows > 0 else True)
    and (f1s_nometa.mean() <= 0.97)
)
print()
print('VERDICT:', 'TEMIZ (bilinen kontrollerin hepsi gecti)' if clean_verdict else 'INCELEME GEREKLI (asagidaki maddelere bak)')

if f1s_nometa.mean() > 0.97:
    print()
    print('!!! ONEMLI: Meta-predictor TAMAMEN cikarilsa bile CV F1 hala '
          f'{f1s_nometa.mean():.4f} -- kirmizi bayragin kaynagi meta-predictor DEGIL.')
    print('    UNKNOWN_INVESTIGATE listesindeki sutunlar (varity_r/vest/rankscore ailesi disinda kalanlar,')
    print('    arrvars__lqt_penetrance, brca1_func_assay__score gibi az-dolu-ama-guclu anotasyonlar,')
    print('    veya populasyon frekans/konservasyon skorlarinin KOMBINASYONU) muhtemel aciklama.')
    print('    Bu, 63k missense verisinin dogal olarak CADD/REVEL/AlphaMissense-benzeri guclu tek-basina')
    print('    tahminciler icermesinden kaynaklaniyor olabilir (sizinti degil, mesru yuksek sinyal) --')
    print('    ADIM 3 (NB52) A6 ablasyonunda bu ayrimi kesinlestirecek.')

from fpdf import FPDF

class NB51Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB51 - Sizinti Denetimi & Saglamlik Kontrolu Raporu', ln=True, align='C')
        self.ln(2)

    def section(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 8, title, ln=True)
        self.set_font('Helvetica', '', 10)

    def kv_table(self, d):
        for k, v in d.items():
            self.cell(0, 6, f'{k}: {v}', ln=True)
        self.ln(2)

report = NB51Report()
report.add_page()

report.section('1. Hizli Sinyal Testi')
report.kv_table({
    'CV F1 (tum feature, meta dahil)': f'{f1s.mean():.4f} +/- {f1s.std():.4f}',
    'CV AUC': f'{aucs.mean():.4f} +/- {aucs.std():.4f}',
    'CV F1 (meta TAMAMEN haric)': f'{f1s_nometa.mean():.4f}',
    'Floor-F1': f'{floor_f1:.4f}',
    'Kirmizi bayrak (F1>0.97 veya AUC>0.995)': RED_FLAG,
})

report.section('2. Tek-Sutun AUC Taramasi')
cat_counts = classification.value_counts().to_dict()
report.kv_table({'AUC>0.95 sutun sayisi': len(high_auc), **{k: int(v) for k, v in cat_counts.items()}})

report.section('3. Meta-Predictor Ablasyonu')
report.kv_table({
    'CV F1 (meta dahil)': f'{f1s.mean():.4f}',
    'CV F1 (meta haric)': f'{f1s_nometa.mean():.4f}',
    'Fark': f'{f1_delta:.4f}',
    'Karar': 'Modelin isini yapiyor, A6ada acikca belgelenmeli' if f1_delta > 0.05 else 'Sinirli katki',
})

report.section('4. Gen-Ezberi Testi')
report.kv_table({
    'CV F1 (rastgele split)': f'{f1s_nometa.mean():.4f}',
    'CV F1 (gen-holdout)': f'{f1s_group.mean():.4f}',
    'Fark': f'{gene_gap:.4f}',
    'Benzersiz gen sayisi': int(groups.nunique()),
    'Karar': 'Ciddi gen ezberi -- gen-holdout birincil metrik' if gene_gap > 0.10 else 'Belirgin degil, yine de raporlanmali',
})

report.section('5. Duplicate Satir Kontrolu')
report.kv_table({
    'Tam kopya varyant sayisi': n_dup_rows,
    'Celiskili etiketli grup': int(len(conflicting_dups)) if n_dup_rows > 0 else 0,
})

report.section('6. Genel Sonuc')
report.kv_table({'Verdict': 'TEMIZ' if clean_verdict else 'INCELEME GEREKLI (meta-haric CV F1 hala yuksek -- ADIM 3 A6 netlestirecek)'})

REPORT_PATH = os.path.join(REPORTS_DIR, 'nb51_leakage_audit.pdf')
report.output(REPORT_PATH)
print(f'PDF rapor yazildi: {REPORT_PATH}')

=== NB51 OZET ===
cv_f1_all_features_incl_meta: 0.9899463898955363
cv_auc_all_features_incl_meta: 0.9995508741553346
red_flag_quick_signal: True
n_high_auc_gt_095: 30
n_high_auc_unknown_investigate: 1
meta_ablation_f1_delta: 0.0021476100316153435
gene_memorization_gap: 0.0055510337295848755
n_duplicate_rows: 18
n_conflicting_label_groups: 0
floor_f1: 0.5429909668785546

VERDICT: INCELEME GEREKLI (asagidaki maddelere bak)

!!! ONEMLI: Meta-predictor TAMAMEN cikarilsa bile CV F1 hala 0.9878 -- kirmizi bayragin kaynagi meta-predictor DEGIL.
    UNKNOWN_INVESTIGATE listesindeki sutunlar (varity_r/vest/rankscore ailesi disinda kalanlar,
    arrvars__lqt_penetrance, brca1_func_assay__score gibi az-dolu-ama-guclu anotasyonlar,
    veya populasyon frekans/konservasyon skorlarinin KOMBINASYONU) muhtemel aciklama.
    Bu, 63k missense verisinin dogal olarak CADD/REVEL/AlphaMissense-benzeri guclu tek-basina
    tahminciler icermesinden kaynaklaniyor olabilir (sizinti degil, mesru yuksek sinyal) -

## Sonraki Adim

**ADIM 3 — NB52 (`notebooks/52_63k_baseline_hypotheses.ipynb`):** H1-H8 hipotez
ablasyonlari (model ailesi, missing stratejisi, sinif dengeleme, feature seti, etiket
kalitesi, meta-predictor dahil/haric -- bu notebook'un A6 bulgusuna gore, feature
engineering). Test seti bu notebook'ta da HALA acilmadi; tum kararlar 5-fold CV'den.